In [6]:
%pip install -q sentence-transformers scikit-learn pandas openpyxl joblib



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
'''MODEL:sentence-transformers/all-MiniLM-L6-v2:'''
import re, joblib, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

XLSX = "ardook_grants_synthetic.xlsx"       # path to your spreadsheet
PROFILE_PATH = "ardook-profile.md"
LABELS = ["Weak", "Moderate", "Strong", "Excellent"]   # ordered low -> high

# 1) Load data. Inputs = what a real grant listing would contain
#    (type + core requirements). Reasons/disqualifiers/priority are NOT used (label leakage).
df = pd.read_excel(XLSX, sheet_name="Grants")
df["text"] = df["Type of Grant"].str.strip() + ". " + df["Core Requirements"].str.strip()
y = df["Fit for Ardook"].map({l: i for i, l in enumerate(LABELS)}).values
groups = df["Core Requirements"].factorize()[0]   # same template text never spans train and test

# 2) Embed with MiniLM (normalised vectors)
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
encoder.max_seq_length = 256
X = encoder.encode(df["text"].tolist(), batch_size=32, normalize_embeddings=True,
                   show_progress_bar=True)
#Understand what Ardook is about 
profile = open(PROFILE_PATH, encoding="utf-8").read()

def section(md, title):
    m = re.search(rf"##\s+{re.escape(title)}\s*\n(.*?)(?=\n##\s|\Z)", md, re.S)
    return m.group(1).strip() if m else ""

profile_chunks = [c for c in [section(profile, "Mission and the problem it solves"),
                              section(profile, "Keywords for matching")] if c]
assert profile_chunks, "Profile sections not found. Check the headings in the .md file."
P_emb = encoder.encode(profile_chunks, normalize_embeddings=True)

def add_profile_feature(emb):
    """Append each grant's best cosine similarity to the profile as an extra column."""
    return np.hstack([emb, (emb @ P_emb.T).max(axis=1, keepdims=True) * 3])

X_aug = add_profile_feature(X)

# 3) Train the classification head, evaluated with grouped cross-validation
clf = LogisticRegression(C=10, max_iter=2000, class_weight="balanced")
pred = cross_val_predict(clf, X, y, cv=GroupKFold(n_splits=5), groups=groups)
print(classification_report(y, pred, target_names=LABELS, digits=3))
print("Confusion matrix (rows=true, cols=pred):\n", confusion_matrix(y, pred))


# 4) Fit on all data and save
clf.fit(X, y)
joblib.dump({"clf": clf, "labels": LABELS}, "ardook_fit_head.joblib")

# 5) Rule layer for hard lead rules from the profile (embeddings handle these poorly)
def apply_rules(text, label_idx):
    t = text.lower()
    if re.search(r"united states|\bus[- ]only|us resident|\bein\b|us-based|us tax", t) \
       or re.search(r"india|sub-saharan|africa", t) and "armenia" not in t:
        return 0                                   # Weak: not a lead (Lead Rule 3)
    if re.search(r"georgia[- ]only|georgian[- ]registered|georgian resident|entity in georgia", t):
        return min(label_idx, 1)                   # capped at Moderate: conditional (Lead Rule 2)
    return label_idx

def score_grants(items):
    """items: list of (grant_type, requirements_text) -> DataFrame of predictions."""
    texts = [f"{g.strip()}. {r.strip()}" for g, r in items]
    P = clf.predict_proba(encoder.encode(texts, normalize_embeddings=True))
    out = []
    for txt, p in zip(texts, P):
        raw = int(p.argmax())
        out.append({"text": txt,"final_fit": LABELS[apply_rules(txt, raw)],
                    **{f"p_{l}": round(float(p[i]), 3) for i, l in enumerate(LABELS)}})
    return pd.DataFrame(out)

# 6) Try it on synthetic grants
synth = [
    ("Grant", "Open to NGOs and social enterprises in Armenia; supports employment of women caregivers of children with disabilities; 10% co-funding."),
    ("Grant", "Business registered in the United States with a US tax ID; women-owned small businesses only."),
    ("Research Grant", "Biotech startups with patented IP and a university lab partner; TRL 3-6."),
]

#7) Try it on real grants from spreadsheet(Her Agenda Breakthrough Grant, Hey Helen Grant (Visionaries), Credibly Small Business Award)
#Strong,Strong, Moderate
real = [
    ("Grant", "Woman entrepreneur; operating business; newsletter subscription; application fee reported"),
    ("Grant", "Woman; ≥50% ownership; U.S. resident; U.S. for-profit; under $1M annual revenue; U.S. business bank account; $15 fee"),
    ("Prize", "U.S.-based active for-profit; ≥6 months operating; business bank account; ≥$500 revenue on recent bank statement")
]
print(score_grants(synth).to_string(index=False))
print("\n")
print(score_grants(real).to_string(index = False))

Batches: 100%|██████████| 7/7 [00:00<00:00, 28.83it/s]


              precision    recall  f1-score   support

        Weak      0.778     0.845     0.810        58
    Moderate      0.527     0.420     0.468        69
      Strong      0.297     0.256     0.275        43
   Excellent      0.244     0.367     0.293        30

    accuracy                          0.500       200
   macro avg      0.462     0.472     0.461       200
weighted avg      0.508     0.500     0.499       200

Confusion matrix (rows=true, cols=pred):
 [[49  0  9  0]
 [12 29  8 20]
 [ 2 16 11 14]
 [ 0 10  9 11]]
                                                                                                                                         text final_fit  p_Weak  p_Moderate  p_Strong  p_Excellent
Grant. Open to NGOs and social enterprises in Armenia; supports employment of women caregivers of children with disabilities; 10% co-funding. Excellent   0.012       0.141     0.093        0.754
                                         Grant. Business registered in t

Notes for this model:
- Does well at seperating weak from everything else. There were 58 weak grants and 49 of them were labeled as weak and 9 of them labeled as strong
- Mislabled the three real grant examples
- 9 mislabeled as weak, 40 mislabeled as moderate, 32 mislabeled as strong, 19 mislabeled as excellent. Sums to 100 mislabeles =  50% accuracy = model is guessing

Ways to improve:
- Embeddings blurring the eligibility facts together. Make the facts yes/no features
- Fix training data(use real grants to train on instead of synthetic)
- Merge strong and excellent if it is having issues distinguishing since they are pretty similar